# Project 2 — Same MLP in PyTorch (autograd + Adam + DataLoader)
Rebuilds P1 with `nn.Linear / nn.ReLU / CrossEntropyLoss / Adam`. Compares against NumPy.
Investigates `forward / backward / optimizer.step() / zero_grad()` explicitly.

In [ ]:
import os  # os for results dir + thread env vars
import random  # random for seeding
import time  # time for throughput measurement
import numpy as np  # numpy for data handling + seeding
import matplotlib.pyplot as plt  # plots for loss curves
import torch  # core pytorch: tensors + autograd
import torch.nn as nn  # nn: Linear/ReLU/Sequential building blocks
from torch.utils.data import TensorDataset, DataLoader  # dataset + batched loader utilities
SEED = 42  # global seed for reproducibility
random.seed(SEED)  # seed python rng
np.random.seed(SEED)  # seed numpy rng
torch.manual_seed(SEED)  # seed torch rng (init + shuffling)
os.makedirs('./results', exist_ok=True)  # ensure results dir exists
print('torch', torch.__version__, '| cuda?', torch.cuda.is_available())  # log env for the report

In [ ]:
def load_tensors():  # build CPU tensors (Xtr,ytr,Xte,yte) with MNIST->digits fallback
    try:  # prefer real MNIST to match P1
        from torchvision import datasets  # torchvision loader (optional)
        tr = datasets.MNIST(root='/tmp/data', train=True, download=True)  # train split
        te = datasets.MNIST(root='/tmp/data', train=False, download=True)  # test split
        Xtr = tr.data.float().reshape(-1, 784) / 255.0  # flatten+scale to tensor float32
        ytr = tr.targets.long()  # labels as int64 tensor
        Xte = te.data.float().reshape(-1, 784) / 255.0  # same for test
        yte = te.targets.long()  # test labels
        print(f'MNIST: {tuple(Xtr.shape)} / {tuple(Xte.shape)}')  # log shapes
        return Xtr, ytr, Xte, yte  # return torch tensors
    except Exception as e:  # offline fallback
        print('fallback to sklearn digits:', e)  # log reason
        from sklearn.datasets import load_digits  # digits dataset
        from sklearn.model_selection import train_test_split  # splitter
        d = load_digits()  # load 8x8 digits
        X = (d.data / 16.0).astype(np.float32)  # scale to [0,1] float32
        y = d.target.astype(np.int64)  # int labels
        Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)  # split
        return torch.from_numpy(Xtr), torch.from_numpy(ytr), torch.from_numpy(Xte), torch.from_numpy(yte)  # to tensors
Xtr, ytr, Xte, yte = load_tensors()  # execute loader once
N_IN = Xtr.shape[1]  # infer input features (784 or 64)
print('n_in =', N_IN)  # confirm for model construction

In [ ]:
class MLP(nn.Module):  # define model as nn.Module so params are tracked
    def __init__(self, n_in, n_hid=128, n_out=10):  # ctor with same width as P1 for fair compare
        super().__init__()  # must call parent ctor to init module machinery
        self.fc1 = nn.Linear(n_in, n_hid)  # Linear1: learnable W1,b1 (autograd-tracked)
        self.relu = nn.ReLU()  # ReLU: no params, elementwise nonlinearity
        self.fc2 = nn.Linear(n_hid, n_out)  # Linear2: hidden -> logits
    def forward(self, x):  # define forward computation (builds autograd graph)
        return self.fc2(self.relu(self.fc1(x)))  # fc1 -> relu -> fc2, returns raw logits (no softmax: CE does it)
model = MLP(N_IN)  # instantiate model with inferred input dim
print(model)  # print architecture (2 Linear + ReLU)
print('params:', sum(p.numel() for p in model.parameters()))  # count total trainable scalars
loss_fn = nn.CrossEntropyLoss()  # CE with logits: includes log-softmax + NLL internally
opt = torch.optim.Adam(model.parameters(), lr=1e-3)  # Adam optimizer (adaptive LR) over all params

In [ ]:
xb = Xtr[:8]  # take 8-sample micro-batch to dissect autograd
yb = ytr[:8]  # matching labels
opt.zero_grad()  # ZERO_GRAD: clear stale .grad buffers (they accumulate by default!)
logits = model(xb)  # FORWARD: run xb through graph, tensors now carry grad_fn
loss = loss_fn(logits, yb)  # compute scalar loss node attached to graph
print('loss =', float(loss))  # show loss value (~2.3 at init)
loss.backward()  # BACKWARD: autograd chain rule fills .grad for every param
print('grad norms:', [round(p.grad.norm().item(), 4) for p in model.parameters()])  # grads now exist; check magnitudes
opt.step()  # STEP: w -= lr*m_hat/(sqrt(v_hat)+eps) inside Adam (uses .grad)
opt.zero_grad()  # clear again so next iteration starts fresh (standard loop hygiene)
print('one manual step done; grads cleared:', all(p.grad is None or p.grad.abs().sum()==0 for p in model.parameters()))  # verify clearing

In [ ]:
BATCH = 64  # mini-batch size (same as P1 for fair comparison)
EPOCHS = 10  # fewer epochs: Adam converges faster than SGD
train_ds = TensorDataset(Xtr, ytr)  # wrap tensors as dataset (indexable)
train_dl = DataLoader(train_ds, batch_size=BATCH, shuffle=True)  # batched+shuffled loader (workers=0 for portability)
test_ds = TensorDataset(Xte, yte)  # test dataset wrapper
test_dl = DataLoader(test_ds, batch_size=512)  # large test batches (no grad needed)
def evaluate():  # helper computing test accuracy without tracking grads
    model.eval()  # set eval mode (dropout/BN would change behavior; none here but good habit)
    correct = total = 0  # init counters
    with torch.no_grad():  # disable autograd: saves memory + time for inference
        for xb, yb in test_dl:  # loop test batches
            pred = model(xb).argmax(1)  # argmax logits -> predicted class
            correct += (pred == yb).sum().item()  # count correct in batch
            total += len(xb)  # count samples
    model.train()  # restore train mode for further training
    return correct / total  # return fraction correct
hist = {'loss': [], 'acc': []}  # log per-epoch loss/acc
t0 = time.time()  # start timer
for epoch in range(1, EPOCHS + 1):  # epoch loop
    model.train()  # ensure train mode
    tot, cnt = 0.0, 0  # weighted loss accumulator + sample counter
    for xb, yb in train_dl:  # mini-batch loop (the DataLoader yields shuffled batches)
        opt.zero_grad()  # clear grads from previous step
        loss = loss_fn(model(xb), yb)  # forward + loss in one line
        loss.backward()  # populate .grad via autograd
        opt.step()  # Adam update
        tot += float(loss) * len(xb)  # weight by batch size
        cnt += len(xb)  # count samples
    acc = evaluate()  # test accuracy this epoch
    hist['loss'].append(tot / cnt)  # log mean loss
    hist['acc'].append(acc)  # log accuracy
    print(f'epoch {epoch:02d}/{EPOCHS} loss={tot/cnt:.4f} test_acc={acc:.4f}')  # progress line
dt = time.time() - t0  # elapsed
print(f'trained in {dt:.1f}s ({len(Xtr)*EPOCHS/dt:.0f} samples/sec)')  # throughput for P1-vs-P2 table

In [ ]:
import csv  # csv for results export
print(f'FINAL test_acc={hist["acc"][-1]:.4f}')  # headline metric
with open('./results/metrics_pytorch.csv', 'w', newline='') as f:  # open results CSV
    w = csv.writer(f)  # writer object
    w.writerow(['epoch', 'loss', 'test_acc'])  # header
    for e, (l, a) in enumerate(zip(hist['loss'], hist['acc']), 1):  # rows
        w.writerow([e, f'{l:.6f}', f'{a:.6f}'])  # write row
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))  # two subplots
ax[0].plot(hist['loss'], marker='o')  # loss curve
ax[0].set(title='PyTorch train loss', xlabel='epoch', ylabel='loss')  # labels
ax[1].plot(hist['acc'], marker='o')  # acc curve
ax[1].set(title='PyTorch test accuracy', xlabel='epoch', ylabel='acc')  # labels
fig.tight_layout()  # layout fix
fig.savefig('./results/curves_pytorch.png', dpi=120)  # save plot
plt.show()  # display
print('Compare with P1 ./results/metrics.csv: PyTorch+Adam should converge in fewer epochs.')  # analysis prompt

## README prompts
Explain `zero_grad` (grad accumulation), what graph `backward` builds, and why Adam beats SGD here. Try SGD(lr=0.1) vs Adam(lr=1e-3) and record the table.